**🌐 Language:** **English** | [한국어 →](/fuzzy-rdd-ko)

# Fuzzy Regression Discontinuity

<small><em>Written by Jiwoo Son · <a href="https://github.com/bungaedm">GitHub</a> · <a href="https://www.linkedin.com/in/jiwoo-son-3207021a7/">LinkedIn</a></em></small>

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings(action='ignore')

In [ ]:
%load_ext autoreload
%autoreload 2

from src.rdd import *
from src.fuzzy_rdd import *
from src.image import *

# Recap

## What is RDD

RDD (Regression Discontinuity Design) is a quasi-experimental method that estimates causal effects by dividing units into treated and untreated groups based on a specific threshold (cutoff) and comparing observations near that threshold.

In [ ]:
show_image('rdd.png')

## Types of RDD (Sharp & Fuzzy)

RDD is broadly divided into Sharp Design and Fuzzy Design.

* `Sharp Design` is when treatment is perfectly determined by whether the running variable crosses the threshold.
* `Fuzzy Design` is when the threshold affects the probability of treatment but does not fully determine actual treatment assignment.

In [ ]:
plot_rdd_type()

# Fuzzy RD

## Summary

Fuzzy RD is used when the probability of treatment changes discontinuously near the cutoff, but treatment assignment is imperfect (i.e., not everyone complies). In this case, we use 2SLS to estimate the LATE (Local Average Treatment Effect).

In [ ]:
plot_fuzzy_rdd(true_effect=0.3)

In [ ]:
show_image('rdd_fuzzy_01.png')

## Basic Setup

- $X_i$: continuous running variable
- $c$: cutoff
- $D_i$: actual treatment status (binary, $D_i \in \{0, 1\}$)
- $Z_i = \mathbf{1}[X_i \geq c]$: instrument variable (indicator for crossing the cutoff)
- $Y_i$: outcome variable

Unlike Sharp RD, in Fuzzy RD the **treatment probability** changes discontinuously at the cutoff:

$$\lim_{x \rightarrow c-} \Pr(D_i = 1 \mid X_i = x) \neq \lim_{x \rightarrow c+} \Pr(D_i = 1 \mid X_i = x)$$

Since the jump in treatment probability is not complete ($\neq 0$ or $1$), we use $Z_i$ as an instrumental variable.

## Identification Assumptions

| Assumption | Description |
|------|------|
| **Continuity** | $\mathbb{E}[Y_i(0) \mid X_i = x]$ and $\mathbb{E}[Y_i(1) \mid X_i = x]$ are continuous at $x = c$ |
| **Relevance** | $Z_i$ has a significant effect on $D_i$: $\alpha_1 \neq 0$ |
| **Exclusion Restriction** | $Z_i$ affects $Y_i$ only through $D_i$ |
| **Monotonicity** | $D_i(Z=1) \geq D_i(Z=0)$ (no defiers) |

## Estimation

### Estimand: LATE

$$\beta_1 = \mathbb{E}[Y_i(1) - Y_i(0) \mid \text{complier}, X_i = c]$$

This is the treatment effect for compliers **near** the cutoff $c$, and may differ from the population-wide ATE.

### 2SLS Estimation: Two Stages

* We perform the analysis using Two-Stage Least Squares (2SLS).
    * First Stage: Use whether the running variable exceeds the cutoff as an instrument to predict actual treatment status.
    * Second Stage: Use the predicted treatment from the first stage to estimate its effect on the outcome variable.

`First Stage`: Treatment regression

$$D_i = \alpha_0 + \alpha_1 Z_i + \alpha_2 (X_i - c) + \alpha_3 Z_i (X_i - c) + \epsilon_i^{(1)}$$

- Estimates the discontinuous effect of $Z_i = \mathbf{1}[X_i \geq c]$ on $D_i$
- Predicted treatment: $\hat{D}_i$
- Key assumption: **Relevance** — $\alpha_1 \neq 0$
    - $\alpha_1$: how much the treatment probability jumps at the cutoff
    - If $\alpha_1 \approx 0$, $Z_i$ barely explains $D_i$, and the variance of the second-stage estimator $\beta_1$ explodes, making it unreliable.

`Second Stage`: Outcome regression

$$Y_i = \beta_0 + \beta_1 \hat{D}_i + \beta_2 (X_i - c) + \beta_3 \hat{D}_i (X_i - c) + \epsilon_i^{(2)}$$

- $\beta_1$: estimated **LATE (Local Average Treatment Effect)**
- Causal effect for compliers near the cutoff $c$

### Expression as the Wald Estimator

The 2SLS estimator is equivalent to the **Wald estimator** at the cutoff:

$$\hat{\tau}_{Fuzzy} = \frac{\text{ITT}_Y}{\text{ITT}_D} = \frac{\lim_{x \rightarrow c+} \mathbb{E}[Y_i \mid X_i = x] - \lim_{x \rightarrow c-} \mathbb{E}[Y_i \mid X_i = x]}{\lim_{x \rightarrow c+} \mathbb{E}[D_i \mid X_i = x] - \lim_{x \rightarrow c-} \mathbb{E}[D_i \mid X_i = x]}$$

- **$\text{ITT}_Y$**: effect of $Z_i$ on the **outcome** $Y_i$

$$\text{ITT}_Y = \lim_{x \downarrow c} \mathbb{E}[Y_i \mid X_i = x] - \lim_{x \uparrow c} \mathbb{E}[Y_i \mid X_i = x]$$

- **$\text{ITT}_D$**: effect of $Z_i$ on **actual treatment** $D_i$

$$\text{ITT}_D = \lim_{x \downarrow c} \mathbb{E}[D_i \mid X_i = x] - \lim_{x \uparrow c} \mathbb{E}[D_i \mid X_i = x]$$

For example, suppose only some students who exceeded the scholarship score threshold actually applied for and received the scholarship:

| | Below Threshold | Above Threshold |
|---|---|---|
| Actual take-up rate | 10% | 60% |
| Grade improvement | 2 points | 5 points |

- $\text{ITT}_D = 0.6 - 0.1 = 0.5$ (jump in treatment probability)
- $\text{ITT}_Y = 5 - 2 = 3$ points (jump in grades)
- $\text{LATE} = \dfrac{3}{0.5} = 6$ points $\leftarrow$ **effect for those who actually complied**

## Key Considerations

* Visualization: It is important to plot a scatter plot with regression lines to check for a discontinuous jump at the cutoff.
* Continuity check: Verify that covariates (age, gender, etc.) are similar just above and below the cutoff.
* Bandwidth selection: Data far from the cutoff should receive less weight, so an appropriate range must be set.

In [ ]:
# Effect of different bandwidth choices
plot_rdd_kernel(bandwidth=1, kernel='triangular', show_kernel=False)
plot_rdd_kernel(bandwidth=3, kernel='triangular', show_kernel=False)
plot_rdd_kernel(bandwidth=5, kernel='triangular', show_kernel=False)

## Advantages and Disadvantages

* Advantages
    * Enables powerful causal inference in real-world settings (policies, promotions, etc.) where random assignment (RCT) is not feasible.
* Disadvantages/Limitations:
    * Local effect: Only captures the effect for units near the cutoff; generalization to the full population may be difficult.
    * Manipulation: If individuals can manipulate the assignment variable (e.g., scores) to cross the cutoff, causal inference becomes invalid.

# Code Example

In [ ]:
plot_fuzzy_rdd(true_effect=0.8)

# Application Examples

## Case 1: Airline Route Analysis

| Item | Description |
|------|------|
| **Research Question** | Causal effect of inter-city distance on the probability of a non-stop flight operating |
| **Running Variable** | Distance between the airports of two cities |
| **Cutoff** | 6,000 miles |
| **Treatment** | Whether a non-stop flight operates on that route |

### Key Points

When the distance between two cities is less than 6,000 miles, the probability of a non-stop flight is significantly higher,
but it is not determined with 100% certainty.
That is, the treatment probability changes discontinuously at the cutoff but is not fully determined —
this is a **Fuzzy RD design**.

### Analysis Method (2SLS)

- **First Stage**: Use the 6,000-mile cutoff as instrument $Z_i$ to predict non-stop flight status $D_i$
$$D_i = \alpha_0 + \alpha_1 Z_i + \alpha_2 (X_i - c) + \epsilon_i^{(1)}$$

- **Second Stage**: Use predicted $\hat{D}_i$ to estimate the causal effect on the outcome variable
$$Y_i = \beta_0 + \beta_1 \hat{D}_i + \beta_2 (X_i - c) + \epsilon_i^{(2)}$$

- A **control function** for the running variable is also included in the model

## Case 2: Public Project Budget and Audit Analysis

| Item | Description |
|------|------|
| **Research Question** | Causal effect of public project budget size on increased monitoring and auditing |
| **Running Variable** | Budget size of the public project |
| **Cutoff** | A specific budget threshold (SAT) |
| **Treatment** | Whether the project is subject to monitoring and auditing |

### Key Points

When a project's budget exceeds a certain cutoff, the likelihood of an audit increases,
but exceeding the budget threshold does not guarantee a 100% audit rate.
Whether a project is audited varies by its nature, making this a **Fuzzy RD design** as well.

### Analysis Method (2SLS)

- **First Stage**: Use whether the budget exceeds the SAT cutoff as an instrument to predict audit status
$$D_i = \alpha_0 + \alpha_1 \mathbf{1}[X_i \geq \text{SAT}] + \alpha_2 (X_i - \text{SAT}) + \epsilon_i^{(1)}$$

- **Second Stage**: Estimate causal effect based on predicted $\hat{D}_i$
$$Y_i = \beta_0 + \beta_1 \hat{D}_i + \beta_2 (X_i - \text{SAT}) + \epsilon_i^{(2)}$$

# Additional Topics

## Weak Instrument Diagnostics

### F-statistic (Rule of Thumb)

Check the F-statistic for the $Z_i$ coefficient in the first-stage regression:

$$F = \frac{\hat{\alpha}_1^2 / \text{Var}(\hat{\alpha}_1)}{1}$$

- $F > 10$: conventionally regarded as a strong instrument (Staiger & Stock, 1997)
- $F < 10$: suspect weak instrument → estimation results are unreliable

### Interpretation in the Fuzzy RD Context

Size of the jump in treatment probability:

$$\hat{\Delta}_D = \lim_{x \downarrow c} \hat{\Pr}(D=1 \mid X=x) - \lim_{x \uparrow c} \hat{\Pr}(D=1 \mid X=x) = \hat{\alpha}_1$$

- **Larger** jump (e.g., ≥ 0.5) → strong instrument
- **Smaller** jump (e.g., < 0.1) → risk of weak instrument

## Methodology Selection Flowchart

### Decision Flow

```
Goal: Causal Inference,
and is random assignment possible?
│
├── YES → Randomized Controlled Trial (RCT)
│
└── NO → Is a quasi-experimental design feasible?
          │
          ├── YES → Are treatment and control groups observed?
          │         │
          │         ├── YES → Is panel data available before and after treatment?
          │         │         │
          │         │         ├── YES → Does the parallel trends assumption hold?
          │         │         │         │
          │         │         │         ├── YES → Difference-in-Differences (DiD) ✅
          │         │         │         │
          │         │         │         └── NO → DiD + Matching  or  Synthetic Control ✅
          │         │         │
          │         │         └── NO → Is treatment assigned by an arbitrary threshold?
          │         │                   │
          │         │                   ├── YES → Regression Discontinuity (RDD) ✅
          │         │                   │
          │         │                   └── NO → (not applicable)
          │         │
          │         └── NO → Is time-series data available before and after treatment?
          │                   │
          │                   ├── YES → Interrupted Time-Series Analysis ✅
          │                   │
          │                   └── NO → Not feasible ❌
          │
          └── NO → Does an instrument variable (IV) exist that induces treatment?
                    │
                    ├── YES → Two-Stage Least Squares (2SLS)
                    │          or Selection Bias Correction Method ✅
                    │
                    └── NO → Is the research setting well-defined?
                              (binary treatment, group definition, sufficient sample)
                              │
                              ├── YES → Matching ✅
                              │
                              └── NO → Regression ✅
```

### Methodology Classification Summary

| Category | Methodology |
|------|--------|
| **Quasi-Experiments** | DiD, DiD + Matching, Synthetic Control, RDD, Interrupted Time-Series |
| **Instrumental Variable** | 2SLS, Selection Bias Correction |
| **Selection on Observables** | Matching, Regression |

## RDD vs DiD vs Synthetic Control Comparison

### Core Comparison Table

| | **RDD** | **DiD** | **Synthetic Control** |
|---|---|---|---|
| **Core Idea** | Quasi-random treatment assignment near the cutoff | Parallel trends assumption for treated and control groups | Construct counterfactual as weighted combination of control units |
| **Identification Strategy** | Discontinuity near the threshold | Difference-in-differences across time × treatment groups | Construct synthetic control using pre-treatment data |
| **Required Data** | Cross-sectional data is sufficient | Panel data (before and after treatment) | Panel data (longer pre-treatment period is better) |
| **Number of Treated Units** | No restriction | No restriction | Works even with **1** treated unit |
| **External Validity** | **Low** (estimates LATE near the cutoff only) | Moderate (ATT for the entire treated group) | Moderate (effect for the specific treated unit) |
| **Internal Validity** | **High** (close to quasi-random assignment) | Depends on parallel trends assumption | Depends on pre-treatment fit |
| **Key Assumption** | Continuity near cutoff, no manipulation | Parallel trends | Treated unit within convex hull of controls |
| **Testability of Assumptions** | Relatively easy (McCrary test, etc.) | Partially verified via pre-trend tests | Partially verified via pre-treatment fit |
| **Sample Efficiency** | **Low** (only observations near cutoff are used) | Uses full sample | Uses full pre-treatment period |
| **Best Applied When** | A clear rule or threshold exists | Before-and-after policy comparison is feasible | Few treated units |

### Unique Strengths of RDD

#### Assumptions are transparent and testable
The parallel trends assumption in DiD is an **assumption about counterfactuals** and is fundamentally untestable.
In contrast, RDD can be **directly verified with observable data** as follows:

| Verification Method | Content |
|-----------|------|
| **McCrary (2008) Density Test** | Check whether the distribution of the running variable is discontinuous at the cutoff (test for manipulation) |
| **Placebo Cutoff Test** | There should be no discontinuity at fake cutoffs |
| **Covariate Smoothness Test** | Covariates should be continuous at the cutoff |

#### Closest to a quasi-experimental design
Near the cutoff, units differ only marginally in their running variable value,
so treated and control units can be seen as **nearly identical**.
This is the observational study design most similar to random assignment (RCT).

$$\text{RCT} \approx \text{RDD} > \text{DiD} \approx \text{Synthetic Control}$$

(in terms of credibility for causal inference)

#### No panel data required
DiD and Synthetic Control require **time-series data before and after treatment**,
but RDD can estimate causal effects with just **a single cross-section**.

### Limitations of RDD (trade-offs)

| Limitation | Description |
|------|------|
| **Limited external validity** | Estimates LATE for compliers near the cutoff → difficult to generalize to the full population |
| **Sample inefficiency** | Observations far from the cutoff contribute little to the analysis |
| **Requires a cutoff** | Cannot be applied without a clear threshold |
| **Manipulation risk** | Units may be aware of the cutoff and have incentives to manipulate the running variable |

### Methodology Selection Guide

```
Does a clear threshold or rule exist?
        ├── YES → Consider RDD first
        └── NO
              ├── Are there few treated units (1–5)?
              │         └── YES → Synthetic Control
              └── Panel data + multiple treated units?
                          └── YES → DiD
```

# References

* [Causal Inference in Data Science (YouTube)](https://www.youtube.com/watch?v=8SIoMJTmO3A)